DATA PIPELINE

In [ ]:


# from https://github.com/goombalab/hnet/blob/main/hnet/utils/tokenizers.py


import numpy as np


class ByteTokenizer:
    def __init__(self):
        self.vocab_size = 256
        self.bos_idx = 254
        self.eos_idx = 255
        self.dtype = np.uint8

    def encode(self, seqs: list[str], add_bos: bool = False, add_eos: bool = False, **kwargs) -> list[dict[str, np.ndarray]]:
        total_outputs = []
        for text in seqs:
            text_byte = text.encode("utf-8")

            if add_bos:
                text_byte = bytes([self.bos_idx]) + text_byte
            if add_eos:
                text_byte = text_byte + bytes([self.eos_idx])
            text_byte = bytearray(text_byte)
            text_byte_ids = np.array(text_byte, dtype=self.dtype)

            total_outputs.append({"input_ids": text_byte_ids})

        return total_outputs

    def decode(self, tokens: np.ndarray | list[int], **kwargs) -> str:
        if isinstance(tokens, np.ndarray):
            tokens = tokens.tolist()
        return bytearray(tokens).decode("utf-8", **kwargs)

In [ ]:
Cog. Arch.    

In [ ]:

from cog_arch.models import ... 

# local encoder / Energy CDiT

from cog_arch.configs import ...

<!-- METALEARNER -->

Training

In [ ]:
# Basado en:
# https://github.com/IDSIA/recurrent-fwp/blob/master/algorithmic/main.py



import os
import time
import random
import argparse
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from data import LTEDataset
from metalearner import FastRNNModel
from eval_utils import compute_accuracy


# -------------------------
# Configuración básica
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


parser = argparse.ArgumentParser("FastRNN LTE - Minimal")
parser.add_argument("--data_dir", type=str, required=True)
parser.add_argument("--level", type=int, default=3, choices=[3, 5, 10])
parser.add_argument("--batch_size", type=int, default=64)
parser.add_argument("--num_epoch", type=int, default=10)
parser.add_argument("--learning_rate", type=float, default=1e-4)
parser.add_argument("--hidden_size", type=int, default=512)
parser.add_argument("--num_layers", type=int, default=2)
parser.add_argument("--seed", type=int, default=1)
parser.add_argument("--work_dir", type=str, default="runs")

args = parser.parse_args()


# -------------------------
# Seeds
# -------------------------
torch.manual_seed(args.seed)
random.seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)


# -------------------------
# Directorio de salida
# -------------------------
run_dir = os.path.join(args.work_dir, time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(run_dir, exist_ok=True)


# -------------------------
# Dataset
# -------------------------
src_pad_idx = 0
tgt_pad_idx = -1

train_data = LTEDataset(
    src_file=f"{args.data_dir}/train_{args.level}.src",
    tgt_file=f"{args.data_dir}/train_{args.level}.tgt",
    src_pad_idx=src_pad_idx,
    tgt_pad_idx=tgt_pad_idx,
)

valid_data = LTEDataset(
    src_file=f"{args.data_dir}/valid_{args.level}.src",
    tgt_file=f"{args.data_dir}/valid_{args.level}.tgt",
    src_pad_idx=src_pad_idx,
    tgt_pad_idx=tgt_pad_idx,
    src_vocab=train_data.src_vocab,
    tgt_vocab=train_data.tgt_vocab,
)

train_loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=args.batch_size)


src_vocab = train_data.src_vocab
tgt_vocab = train_data.tgt_vocab
no_print_idx = tgt_vocab.get_no_op_id()


# -------------------------
# Modelo FastRNN
# -------------------------
model = FastRNNModel(
    in_vocab_size=src_vocab.size(),
    out_vocab_size=tgt_vocab.size(),
    hidden_size=args.hidden_size,
    num_layers=args.num_layers,
    num_head=1,
    dim_head=args.hidden_size,
    dim_ff=4 * args.hidden_size,
    dropout=0.0,
    use_pos_enc=True,
).to(DEVICE)

print(model)
print(f"Trainable params: {model.num_params()}")


# -------------------------
# Optimización
# -------------------------
criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)


# -------------------------
# Entrenamiento
# -------------------------
print("===> Starting training")
best_val_acc = 0.0

for epoch in range(args.num_epoch):
    model.train()
    total_loss = 0.0

    for src, tgt in train_loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        logits = model(src)
        loss = criterion(
            logits.view(-1, logits.size(-1)),
            tgt.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # -------- Validation --------
    model.eval()
    with torch.no_grad():
        val_loss, val_acc, _, _ = compute_accuracy(
            model,
            valid_loader,
            criterion,
            no_print_idx=no_print_idx,
            pad_value=tgt_pad_idx,
            show_example=False,
        )

    print(
        f"[Epoch {epoch+1}] "
        f"train_loss={avg_train_loss:.4f} "
        f"val_loss={val_loss:.4f} "
        f"val_acc={val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(run_dir, "best_model.pt"))


print(f"Best validation accuracy: {best_val_acc:.2f}%")
print("Training finished.")


MVM - Enviroment to execute and generate the Cog Arch  / DAG / etc